# Matemáticas para IA con NumPy y SciPy

[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/LegalIntermediaSL/InteligenciaArtificial/blob/main/tutoriales/notebooks/fundamentos/06-matematicas-para-ia.ipynb)

Álgebra lineal, probabilidad y cálculo aplicados a IA con NumPy y SciPy.

Este notebook recorre los conceptos matemáticos imprescindibles que subyacen a los modelos de lenguaje y al aprendizaje automático en general: vectores, matrices, probabilidad y gradientes. Cada sección muestra cómo la teoría se traduce directamente en código Python.

In [ ]:
# Instalación de dependencias
%pip install numpy scipy matplotlib --quiet

import numpy as np
import scipy.special as sp
import matplotlib.pyplot as plt

print(f'NumPy: {np.__version__}')
print('Entorno listo.')

## 1. Vectores y producto escalar (dot product)

En IA, los vectores representan **embeddings**: cada token, imagen o documento se codifica como un punto en un espacio de alta dimensión. El producto escalar mide cuánto 'se parecen' dos vectores.

In [ ]:
# Embeddings simplificados de dos frases (d=4 para ilustrar)
v_gato = np.array([0.8, 0.1, 0.05, 0.05])
v_perro = np.array([0.75, 0.15, 0.05, 0.05])
v_coche = np.array([0.1, 0.05, 0.8, 0.05])

dot_gato_perro = np.dot(v_gato, v_perro)
dot_gato_coche = np.dot(v_gato, v_coche)

print(f'Dot(gato, perro) = {dot_gato_perro:.4f}')  # Debería ser alto
print(f'Dot(gato, coche) = {dot_gato_coche:.4f}')  # Debería ser bajo

# Norma (longitud) de un vector
norma = np.linalg.norm(v_gato)
print(f'Norma de v_gato: {norma:.4f}')

## 2. Similitud coseno (Cosine Similarity)

La similitud coseno normaliza el producto escalar por la longitud de los vectores, lo que la hace invariante a la escala. Es la métrica estándar en bases de datos vectoriales como Pinecone o ChromaDB.

In [ ]:
def cosine_similarity(a: np.ndarray, b: np.ndarray) -> float:
    """Similitud coseno entre dos vectores."""
    return np.dot(a, b) / (np.linalg.norm(a) * np.linalg.norm(b))

sim_gato_perro = cosine_similarity(v_gato, v_perro)
sim_gato_coche = cosine_similarity(v_gato, v_coche)

print(f'cos_sim(gato, perro) = {sim_gato_perro:.4f}  → muy similar')
print(f'cos_sim(gato, coche) = {sim_gato_coche:.4f}  → poco similar')

# Distancia coseno (usada en ChromaDB por defecto)
dist_coseno = 1 - sim_gato_perro
print(f'Distancia coseno(gato, perro) = {dist_coseno:.4f}')

## 3. Matrices y multiplicación matricial

Las capas de un transformer son fundamentalmente multiplicaciones matriciales. La proyección Q·K^T en la atención es una multiplicación de matrices que calcula cuánto debe 'atender' cada token a cada otro.

In [ ]:
# Simulamos la capa de atención simplificada: Attention(Q,K,V)
np.random.seed(42)
seq_len = 4   # 4 tokens
d_model = 8   # dimensión del modelo
d_k = 4       # dimensión de las proyecciones Q y K

# Matrices de proyección (pesos aprendidos)
W_Q = np.random.randn(d_model, d_k) * 0.1
W_K = np.random.randn(d_model, d_k) * 0.1

# Secuencia de entrada: 4 tokens, cada uno con dimensión d_model
X = np.random.randn(seq_len, d_model)

Q = X @ W_Q   # (4, 4)
K = X @ W_K   # (4, 4)

# Puntuaciones de atención (escaladas)
scores = Q @ K.T / np.sqrt(d_k)   # (4, 4)
print('Forma de Q:', Q.shape)
print('Forma de scores Q·Kᵀ:', scores.shape)
print('Scores (primeras 2 filas):\n', np.round(scores[:2], 3))

## 4. Softmax y divergencia KL

**Softmax** convierte un vector de puntuaciones reales en una distribución de probabilidad (suma 1, todos positivos). Es la función de salida del transformer para predecir el siguiente token.

La **divergencia KL** mide cuánto difiere una distribución de otra — se usa en RLHF para penalizar que el modelo se aleje demasiado del modelo de referencia.

In [ ]:
# Softmax manual vs scipy
logits = np.array([2.0, 1.0, 0.5, -1.0])  # puntuaciones del transformer

def softmax(x: np.ndarray) -> np.ndarray:
    e_x = np.exp(x - np.max(x))  # estabilidad numérica
    return e_x / e_x.sum()

probs = softmax(logits)
print(f'Logits:  {logits}')
print(f'Softmax: {np.round(probs, 4)}')
print(f'Suma: {probs.sum():.6f}  (debe ser 1.0)')

# Divergencia KL: D_KL(P || Q) = Σ P(i) * log(P(i)/Q(i))
# Modelo ajustado vs modelo de referencia (RLHF)
p_ajustado = softmax(logits)
p_referencia = softmax(logits * 0.5)  # distribución más uniforme

kl_div = np.sum(p_ajustado * np.log(p_ajustado / (p_referencia + 1e-10)))
print(f'\nKL(ajustado || referencia) = {kl_div:.4f}')
print('(RLHF: penalizar si KL > umbral para mantener coherencia)')

## 5. Gradiente numérico

El **gradiente** indica la dirección de máximo crecimiento de una función. Durante el entrenamiento, el optimizador mueve los pesos en la dirección **contraria** al gradiente para minimizar la pérdida.

In [ ]:
def cross_entropy_loss(logits: np.ndarray, target_idx: int) -> float:
    """Pérdida cross-entropy para un solo ejemplo."""
    probs = softmax(logits)
    return -np.log(probs[target_idx] + 1e-10)

# Gradiente numérico: df/dx ≈ (f(x+h) - f(x-h)) / 2h
def gradiente_numerico(f, x: np.ndarray, h: float = 1e-5) -> np.ndarray:
    grad = np.zeros_like(x)
    for i in range(len(x)):
        x_plus = x.copy(); x_plus[i] += h
        x_minus = x.copy(); x_minus[i] -= h
        grad[i] = (f(x_plus) - f(x_minus)) / (2 * h)
    return grad

logits = np.array([2.0, 1.0, 0.5, 0.1])
target = 0  # el token correcto es el índice 0

f = lambda x: cross_entropy_loss(x, target)
loss = f(logits)
grad = gradiente_numerico(f, logits)

print(f'Loss inicial: {loss:.4f}')
print(f'Gradiente:    {np.round(grad, 4)}')
print('(Actualización SGD: logits -= lr * grad)')

## 6. Backpropagación en una neurona

La backprop aplica la regla de la cadena para calcular gradientes de forma eficiente. Aquí lo implementamos desde cero para una neurona con activación sigmoide.

In [ ]:
# Forward + backward pass de una neurona: y = sigmoid(w·x + b)
class Neurona:
    def __init__(self):
        self.w = np.array([0.5, -0.3, 0.8])
        self.b = 0.1
    
    def sigmoid(self, z):
        return 1.0 / (1.0 + np.exp(-z))
    
    def forward(self, x):
        self.x = x
        self.z = np.dot(self.w, x) + self.b
        self.y = self.sigmoid(self.z)
        return self.y
    
    def backward(self, d_loss_dy):
        # dσ/dz = σ(z) * (1 - σ(z))
        d_sigma_dz = self.y * (1 - self.y)
        d_loss_dz = d_loss_dy * d_sigma_dz
        # Gradientes respecto a pesos y bias
        d_loss_dw = d_loss_dz * self.x
        d_loss_db = d_loss_dz
        return d_loss_dw, d_loss_db

neurona = Neurona()
x = np.array([1.0, 2.0, -1.0])
y_pred = neurona.forward(x)
y_real = 1.0  # etiqueta binaria

# Pérdida MSE simplificada
loss = 0.5 * (y_pred - y_real) ** 2
d_loss_dy = y_pred - y_real  # gradiente de MSE

grad_w, grad_b = neurona.backward(d_loss_dy)
print(f'Predicción: {y_pred:.4f}  |  Real: {y_real}')
print(f'Loss: {loss:.4f}')
print(f'∂L/∂w: {np.round(grad_w, 4)}')
print(f'∂L/∂b: {grad_b:.4f}')

# Un paso de descenso del gradiente
lr = 0.1
neurona.w -= lr * grad_w
neurona.b -= lr * grad_b
y_nuevo = neurona.forward(x)
print(f'\nPredicción tras un paso GD: {y_nuevo:.4f}  (más cercano a {y_real})')

## 7. Visualización: Softmax y temperatura

La **temperatura** controla la 'confianza' del modelo al generar texto. Temperatura baja → distribución más concentrada (determinista). Temperatura alta → distribución más plana (creativa/aleatoria).

In [ ]:
tokens = ['el', 'un', 'la', 'este', 'aquel']
logits = np.array([3.0, 2.0, 1.5, 0.5, -0.5])
temperaturas = [0.3, 1.0, 2.0]

fig, axes = plt.subplots(1, 3, figsize=(12, 4))
for ax, T in zip(axes, temperaturas):
    probs = softmax(logits / T)
    ax.bar(tokens, probs, color='steelblue', edgecolor='white')
    ax.set_title(f'Temperatura = {T}')
    ax.set_ylabel('Probabilidad')
    ax.set_ylim(0, 1)
    for i, p in enumerate(probs):
        ax.text(i, p + 0.02, f'{p:.2f}', ha='center', fontsize=9)

plt.suptitle('Efecto de la temperatura en Softmax', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()

## 8. Resumen: Conceptos matemáticos y su uso en IA

| Concepto | Fórmula | Dónde se usa en IA |
|---|---|---|
| Producto escalar | `a·b = Σ aᵢbᵢ` | Puntuaciones de atención Q·K |
| Similitud coseno | `a·b / (‖a‖‖b‖)` | Búsqueda vectorial (RAG) |
| Multiplicación matricial | `C = A @ B` | Proyecciones Q, K, V; FFN |
| Softmax | `eˣⁱ / Σeˣʲ` | Distribución sobre vocabulario |
| Divergencia KL | `Σ P log(P/Q)` | RLHF, penalización de deriva |
| Gradiente | `∂L/∂w` | SGD, Adam — actualizar pesos |
| Regla de la cadena | `∂L/∂w = ∂L/∂y · ∂y/∂w` | Backpropagación |
| Temperatura | `softmax(logits/T)` | Sampling, creatividad del LLM |

### Próximos pasos
- [Notebook 07 — Redes Neuronales desde Cero](07-redes-neuronales-scratch.ipynb)
- [Tutorial de APIs](../../apis/01-primeros-pasos.md)
- Profundiza con [3Blue1Brown: Essence of Linear Algebra](https://www.youtube.com/playlist?list=PLZHQObOWTQDPD3MizzM2xVFitgF8hE_ab)